# Calling malignant cells from the inferCNV residuals

Reads the per-cell tables written by run_infercnv.R and does not re-run inferCNV.

- Position of every cell on its cohort's stromal null
- Verdict per epithelial subcluster (median null position), with subclusters computed by inferCNV inside each per-patient run, as Shiao et al. take the verdict per subcluster
- Cohort gate: a cohort with less than 10% of its epithelium above the null has no detectable tumor, and its epithelium becomes not_tested
- Specificity from held-out immune cells, which are not part of the reference
- Final status per cell (malignant, borderline, non_malignant, not_tested)
- Output: cnv_status.csv, plus tables and figures in the 03_1_infercnv folders

Per-patient runs, a T/NK + myeloid reference, B/plasma cells left out, two continuous axes, no HMM.

## Packages and paths

In [1]:
from __future__ import annotations

import h5py
from anndata.io import read_elem
import pandas as pd
import numpy as np
import os
import glob

import matplotlib.pyplot as plt

plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['savefig.bbox'] = 'tight'
plt.rcParams['figure.facecolor'] = 'white'

try:
    SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    SCRIPT_DIR = os.getcwd()
PHASE_DIR = os.path.dirname(SCRIPT_DIR) # 03_drvi_tumoral_epi/

FIG_DIR = os.path.join(PHASE_DIR, 'figures', '03_1_infercnv')
TABLE_DIR = os.path.join(PHASE_DIR, 'tables', '03_1_infercnv')
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TABLE_DIR, exist_ok=True)

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'

print('FIG_DIR  :', FIG_DIR)
print('TABLE_DIR:', TABLE_DIR)

FIG_DIR  : /home/catt/QCB-thesis-final/03_drvi_tumoral_epi/figures/03_1_infercnv
TABLE_DIR: /home/catt/QCB-thesis-final/03_drvi_tumoral_epi/tables/03_1_infercnv


In [2]:
DATA_DIR = os.environ['DATA_DIR'] # export DATA_DIR=<data root> before starting Jupyter
DATA_DIR = os.path.abspath(os.path.expanduser(DATA_DIR))
CNV_DIR = os.path.join(DATA_DIR, '03_tum')
SUMMARY_DIR = os.path.join(CNV_DIR, 'summary')

SHIAO_H5AD = os.path.join(DATA_DIR, 'shiao.h5ad') # only .obs is read
STATUS_OUT = os.path.join(CNV_DIR, 'cnv_status.csv') # output of this notebook

GROUP_EPI = 'epi'
GROUP_STROMAL = 'stromal'
GROUP_HELDOUT = 'immune_heldout'
REF_GROUPS = ('ref_tcell', 'ref_myeloid')

print('output  :', STATUS_OUT)


output  : /home/catt/QCB-thesis-final/datasets/03_tum/cnv_status.csv


## The parameters

The quantile and the minimum null size set the per-cell rule; the subcluster cuts, the minimum subcluster size and the cohort gate set the verdict per subcluster and per cohort.

In [3]:
Q = 0.95 # quantile of the cohort's stromal null for the per-cell rule
MIN_NULL = 30 # fewer stromal cells than this cannot define a null

MAL_MED = 0.90 # subcluster median nullpos >= this -> malignant
NORM_MED = 0.75 # <= this -> normal, in between -> borderline
MIN_SUB = 20 # smaller subclusters get no verdict, their cells fall back to the per-cell rule
GATE_FRAC = 0.10 # fraction of epithelium above the null, on either readout, for a cohort to pass

# True: a cell in a malignant subcluster must also pass the per-cell rule (cell_rule is saved either way)
REQUIRE_CELL_RULE = False


## The per-cell tables

In [4]:
paths = sorted(glob.glob(os.path.join(SUMMARY_DIR, '*_cnv.csv')))
assert paths, f'no summaries in {SUMMARY_DIR}: run ./infercnv_all.sh first'
cnv = pd.concat([pd.read_csv(p) for p in paths], ignore_index=True)
print(f'{len(paths)} cohort(s), {len(cnv):,} cells')

census = pd.read_csv(os.path.join(CNV_DIR, 'cohort_census.csv'))
prepared = set(census.loc[census['status'] == 'prepared', 'cohort'])
ran = set(cnv['cohort'].unique())
print(f'prepared: {len(prepared)} | with a summary: {len(ran)}')
missing = sorted(prepared - ran)
if missing:
    print('PREPARED BUT NO SUMMARY (inferCNV failed on these):', missing)

cnv.groupby('group', observed=True).size()

33 cohort(s), 196,606 cells
prepared: 33 | with a summary: 33


group
epi               75438
immune_heldout    16191
ref_myeloid       30709
ref_tcell         32399
stromal           41869
dtype: int64

In [5]:
# Only .obs is read: backed read_h5ad would still load the counts layer into memory.
with h5py.File(SHIAO_H5AD, 'r') as f:
    shiao_obs = read_elem(f['obs'])[['cohort', 'cell_type']]
assert cnv['cell'].isin(shiao_obs.index).all(), 'a cell in a summary is not in shiao.h5ad'

## Step 1 - position of each cell on its cohort's null

For each cell, p_score and p_corr are the fractions of its cohort's stromal cells it exceeds on the two axes, and nullpos is the smaller of the two. A nullpos above the quantile is the per-cell rule, and the same number is aggregated per subcluster in step 2.

Being a percentile within the cohort, nullpos is comparable across patients, while the raw CNV score is not.

In [6]:
stromal = cnv[cnv['group'] == GROUP_STROMAL]
n_null = stromal.groupby('cohort', observed=True).size()
usable = set(n_null[n_null >= MIN_NULL].index)
unusable = sorted(set(cnv['cohort'].unique()) - usable)
print(f'cohorts with a usable stromal null: {len(usable)}')
print(f'cohorts without one (all their cells -> not_tested): {unusable if unusable else "none"}')

parts = []
for cohort, d in cnv.groupby('cohort', observed=True):
    s = stromal[stromal['cohort'] == cohort]
    if len(s) < MIN_NULL:
        continue
    d = d.copy()
    for ax in ('cnv_score', 'cnv_corr'):
        d['p_' + ax] = np.searchsorted(np.sort(s[ax].to_numpy()), d[ax].to_numpy()) / len(s)
    parts.append(d)

a = pd.concat(parts, ignore_index=True)
a['nullpos'] = np.minimum(a['p_cnv_score'], a['p_cnv_corr'])
a['cell_rule'] = a['nullpos'] > Q
print(f'\n{len(a):,} cells on a usable null')

cohorts with a usable stromal null: 33
cohorts without one (all their cells -> not_tested): none

196,606 cells on a usable null


In [7]:
# Crossing rate per group; the stromal rate is the null's own false-positive rate, used by the gate.
a.groupby('group', observed=True)['cell_rule'].agg(['size', 'sum', 'mean']).round(4)

,size,sum,mean
group,,,
epi,75438,36495,0.4838
immune_heldout,16191,45,0.0028
ref_myeloid,30709,105,0.0034
ref_tcell,32399,26,0.0008
stromal,41869,302,0.0072


## Step 2 - verdict per epithelial subcluster

Subclusters are computed by inferCNV inside each per-patient run (leiden, resolution 0.005), so they never mix patients and each one is a candidate clone. They are computed on the same residuals they are judged on, so leiden will split any patient into higher and lower groups; the threshold still comes from the stromal null, and the cohort gate in step 3 stops a single high subcluster from making a tumor.

A subcluster is malignant when its median null position is at least 0.90, normal when at most 0.75, borderline in between. Subclusters under 20 cells get no verdict (undecided). The resolution sweep (sensitivity_resolution.py) gives the same per-patient verdict from 0.005 to 0.05; the coarse end keeps most cells in subclusters large enough to take a median over.

In [8]:
# Subclusters come with the summaries, computed inside each per-patient inferCNV run.
assert 'subcluster' in a.columns, (
    'the summaries have no `subcluster` column: they were produced by the old '
    "analysis_mode='samples' runs. Re-run ./infercnv_all.sh")
a['subcluster'] = a['subcluster'].astype('object')
a.loc[a['group'] != GROUP_EPI, 'subcluster'] = np.nan
# Subcluster names repeat across runs, so the key is (cohort, subcluster).
a['sub_id'] = np.where(a['subcluster'].isna(), np.nan,
                       a['cohort'].astype(str) + '|' + a['subcluster'].astype(str))

epi = a[a['group'] == GROUP_EPI]
n_unlab = int(epi['subcluster'].isna().sum())
print(f'epithelial cells: {len(epi):,} | without a subcluster: {n_unlab:,}')
print(f"subclusters: {epi['sub_id'].nunique():,} over {epi['cohort'].nunique()} cohorts "
      f"(median {int(epi.groupby('sub_id', observed=True).size().median())} cells)")


epithelial cells: 75,438 | without a subcluster: 0
subclusters: 786 over 33 cohorts (median 66 cells)


In [9]:
g = epi.dropna(subset=['sub_id']).groupby('sub_id', observed=True)
clusters = pd.DataFrame({'cohort': g['cohort'].first(), 'subcluster': g['subcluster'].first(),
                         'n': g.size(), 'median_nullpos': g['nullpos'].median(),
                         'frac_cell_rule': g['cell_rule'].mean()})
# No verdict below MIN_SUB: a median over a handful of cells is not an aggregate.
clusters['too_small'] = clusters['n'] < MIN_SUB
clusters['call'] = np.where(
    clusters['too_small'], 'undecided',
    np.where(clusters['median_nullpos'] >= MAL_MED, 'malignant',
             np.where(clusters['median_nullpos'] <= NORM_MED, 'normal', 'borderline')))
clusters = clusters.sort_values(['cohort', 'median_nullpos'])
clusters.to_csv(os.path.join(TABLE_DIR, 'epi_subcluster_call.csv'))

print(clusters['call'].value_counts().to_string())
print(f"\ncells in subclusters too small to decide: "
      f"{int(clusters.loc[clusters['too_small'], 'n'].sum()):,} "
      f"({clusters.loc[clusters['too_small'], 'n'].sum() / len(epi):.1%} of the epithelium)")
clusters.head(20).round(3)


call
malignant     317
normal        239
undecided     199
borderline     31

cells in subclusters too small to decide: 900 (1.2% of the epithelium)


,cohort,subcluster,n,median_nullpos,frac_cell_rule,too_small,call
sub_id,,,,,,,
Patient01|epi_s2,Patient01,epi_s2,173,0.288,0.023,False,normal
Patient01|epi_s1,Patient01,epi_s1,335,0.999,0.925,False,malignant
Patient01|epi_s3,Patient01,epi_s3,34,1.000,0.971,False,malignant
Patient02|epi_s1,Patient02,epi_s1,209,0.549,0.120,False,normal
Patient03|epi_s2,Patient03,epi_s2,145,0.739,0.041,False,normal
Patient03|epi_s1,Patient03,epi_s1,89,0.997,0.955,False,malignant
Patient04|epi_s1,Patient04,epi_s1,192,0.417,0.042,False,normal
Patient05|epi_s2,Patient05,epi_s2,282,0.296,0.011,False,normal
Patient05|epi_s1,Patient05,epi_s1,94,0.996,0.915,False,malignant


## Step 3 - the cohort gate

If a cohort's epithelium is indistinguishable from its stroma, the cells above the cut are the tail of the null, not tumor. A cohort passes when at least 10% of its epithelium is above the null on either readout: per cell, or in a malignant subcluster. Either rather than both, because a noisy stromal block (e.g. Patient15) can suppress the per-cell readout alone.

The epithelium of a failed cohort becomes not_tested, not non_malignant.

In [10]:
gate = epi.groupby('cohort', observed=True).agg(
    n_epi=('cell_rule', 'size'),
    frac_cell_rule=('cell_rule', 'mean'),
    n_subclusters=('sub_id', lambda s: int(s.nunique())))
mal_subs = set(clusters.index[clusters['call'] == 'malignant'])
gate['frac_malig_subcluster'] = (epi.assign(hit=epi['sub_id'].isin(mal_subs))
                                 .groupby('cohort', observed=True)['hit'].mean())
gate['stromal_rate'] = a[a['group'] == GROUP_STROMAL].groupby('cohort', observed=True)['cell_rule'].mean()
# Specificity: held-out immune cells are not in the reference and set no threshold,
# unlike the reference cells, which sit at ~0 by construction.
gate['heldout_rate'] = a[a['group'] == GROUP_HELDOUT].groupby('cohort', observed=True)['cell_rule'].mean()
gate['ref_rate'] = a[a['group'].isin(REF_GROUPS)].groupby('cohort', observed=True)['cell_rule'].mean()
gate['pass'] = ((gate['frac_cell_rule'] >= GATE_FRAC)
                | (gate['frac_malig_subcluster'].fillna(0) >= GATE_FRAC))
gate.to_csv(os.path.join(TABLE_DIR, 'cohort_gate.csv'))

print(f"cohorts passing the gate: {int(gate['pass'].sum())} of {len(gate)}")
_sick = gate[gate['heldout_rate'] > gate['stromal_rate']]
if len(_sick):
    print('\nSPECIFICITY WARNING - held-out immune crossing above the stromal rate:')
    print(_sick[['heldout_rate', 'stromal_rate', 'n_epi']].round(4).to_string())
print('\nFAILED - no detectable tumor, their epithelium becomes not_tested:')
gate[~gate['pass']].round(4)


cohorts passing the gate: 26 of 33

SPECIFICITY WARNING - held-out immune crossing above the stromal rate:
           heldout_rate  stromal_rate  n_epi
cohort                                      
Patient26        0.0680        0.0137   5815
Patient48        0.0094        0.0055   1192
Patient66        0.0020        0.0000    214

FAILED - no detectable tumor, their epithelium becomes not_tested:


,n_epi,frac_cell_rule,n_subclusters,frac_malig_subcluster,stromal_rate,heldout_rate,ref_rate,pass
cohort,,,,,,,,
Patient04,192,0.0417,1,0.0000,0.0026,0.0,0.000,False
Patient07,257,0.0895,2,0.0000,0.0000,0.0,0.000,False
Patient15,549,0.0401,2,0.0000,0.0227,0.0,0.001,False
Patient23,176,0.0341,1,0.0000,0.0015,0.0,0.000,False
Patient39,1633,0.0423,14,0.0429,0.0025,0.0,0.000,False
Patient52,9243,0.0334,143,0.0162,0.0095,0.0,0.000,False
Patient57,1509,0.0398,11,0.0000,0.0096,0.0,0.000,False


## Step 4 - combining the two verdicts

| subcluster | per-cell rule | status |
|---|---|---|
| malignant | passes | malignant |
| malignant | fails | malignant (borderline if the per-cell rule is required) |
| borderline | either | borderline |
| normal | passes | borderline |
| normal | fails | non_malignant |
| undecided or none | passes | malignant |
| undecided or none | fails | non_malignant |
| any, in a cohort that failed the gate | | not_tested |

Borderline cells are those where the two criteria disagree.

In [11]:
a['cohort_pass'] = a['cohort'].map(gate['pass']).fillna(False)
a['subcluster_call'] = a['sub_id'].map(clusters['call'])

in_mal = a['subcluster_call'] == 'malignant'
if REQUIRE_CELL_RULE:
    in_mal = in_mal & a['cell_rule']

# First matching condition wins: non-epithelial and gated-out cells are settled first.
status = np.select(
    [a['group'] != GROUP_EPI,
     ~a['cohort_pass'],
     in_mal,
     a['subcluster_call'] == 'borderline',
     (a['subcluster_call'] == 'malignant') & ~a['cell_rule'], # only if REQUIRE_CELL_RULE
     (a['subcluster_call'] == 'normal') & a['cell_rule'],
     # undecided or no subcluster: fall back to the per-cell rule
     a['subcluster_call'].isin(['undecided']) & a['cell_rule'],
     a['subcluster_call'].isna() & a['cell_rule']],
    ['non_malignant', 'not_tested', 'malignant', 'borderline', 'borderline', 'borderline',
     'malignant', 'malignant'],
    default='non_malignant')
a['cnv_status'] = status

epi_new = a[a['group'] == GROUP_EPI]
print('epithelial cells by status:')
print(epi_new['cnv_status'].value_counts().to_string())
print('\nof which decided by the per-cell fallback (subcluster too small or absent):')
_fb = epi_new[epi_new['subcluster_call'].isin(['undecided']) | epi_new['subcluster_call'].isna()]
print(f'  {len(_fb):,} cells ({len(_fb)/len(epi_new):.1%})')


epithelial cells by status:
cnv_status
malignant        42291
non_malignant    16839
not_tested       13559
borderline        2749

of which decided by the per-cell fallback (subcluster too small or absent):
  900 cells (1.2%)


## What gets written

One row per cell of shiao.h5ad, as in cnv_status.csv, with the intermediate columns kept so the call can be re-decided downstream:

| column | content |
|---|---|
| cnv_status | malignant, borderline, non_malignant or not_tested |
| cnv_score, cnv_corr | the two raw axes |
| p_score, p_corr, nullpos | position on the cohort's stromal null |
| cell_rule | whether the per-cell rule passes |
| subcluster, subcluster_call | inferCNV subcluster and its verdict |
| cohort_gate | whether the cohort passed the gate |

Subcluster names repeat in every run, so they must be read together with the cohort. Cells no run covered are not_tested, with the other columns empty.

In [13]:
all_cells = shiao_obs.index

cols = ['cnv_status', 'cnv_score', 'cnv_corr', 'p_cnv_score', 'p_cnv_corr', 'nullpos',
        'cell_rule', 'subcluster', 'subcluster_call', 'cohort_pass']
out = (a.set_index('cell')[cols]
       .rename(columns={'p_cnv_score': 'p_score', 'p_cnv_corr': 'p_corr',
                        'cohort_pass': 'cohort_gate'})
       .reindex(all_cells))
out['cnv_status'] = out['cnv_status'].fillna('not_tested')
out.index.name = 'cell'

assert len(out) == len(all_cells) and out['cnv_status'].notna().all()
out.to_csv(STATUS_OUT)
print(f'wrote {STATUS_OUT}  ({len(out):,} rows)')
out['cnv_status'].value_counts()


wrote /home/catt/QCB-thesis-final/datasets/03_tum/cnv_status.csv  (619,693 rows)


cnv_status
not_tested       436646
non_malignant    138007
malignant         42291
borderline         2749
Name: count, dtype: int64

## Malignant cells per patient

Epithelial cells of every patient by status, in cohort order. Patients with fewer than 100 malignant cells are greyed out: they do not enter the tumor subset of 03_2. Patient06 was never run (14 epithelial cells), so its epithelium is all not_tested. The same counts are written to per_cohort_call.csv.

In [ ]:
import sys
from matplotlib.patches import Patch

# Epithelial set of the input preparation, so that Patient06 (never run) is counted too.
sys.path.insert(0, SCRIPT_DIR)
from prepare_infercnv_input import EPITHELIAL

MIN_MAL = 100 # MIN_CELLS_PER_COHORT of utils/cell_set.py: a patient's cells enter the tumor subset
STATUS_ORDER = ['malignant', 'borderline', 'non_malignant', 'not_tested']
STATUS_COLORS = {'malignant': '#2f7d4f', 'borderline': '#e0a84f',
                 'non_malignant': '#5f7fb0', 'not_tested': '#cfcfcf'}
EXCLUDED_COLOR = '#9a9a9a'

epi_all = shiao_obs[shiao_obs['cell_type'].astype(str).isin(EPITHELIAL)].join(out['cnv_status'])
counts = (epi_all.groupby(['cohort', 'cnv_status'], observed=False).size().unstack(fill_value=0)
          .reindex(columns=STATUS_ORDER, fill_value=0))
keep = counts['malignant'] >= MIN_MAL
# Same counts as the plot, so Patient06 (never run) has its row too.
counts.assign(kept_03_2=keep).to_csv(os.path.join(TABLE_DIR, 'per_cohort_call.csv'))
print(f"cohorts with >={MIN_MAL} malignant cells: {int(keep.sum())}"
      f"  |  cells: {counts.loc[keep, 'malignant'].sum():,}")

fig, ax = plt.subplots(figsize=(11, 4.2))
x = np.arange(len(counts))
bottom = np.zeros(len(counts))
for s in STATUS_ORDER:
    colors = [EXCLUDED_COLOR if s == 'malignant' and not k else STATUS_COLORS[s] for k in keep]
    ax.bar(x, counts[s], bottom=bottom, color=colors, width=.8, edgecolor='white', linewidth=.4,
           hatch='////' if s == 'not_tested' else None)
    bottom += counts[s].to_numpy()
ax.set_xticks(x); ax.set_xticklabels(counts.index, rotation=90, fontsize=8)
for lab, k in zip(ax.get_xticklabels(), keep):
    if not k:
        lab.set_color('0.55')
ax.set_xlim(-.6, len(counts) - .4)
ax.set_ylabel('epithelial cells')
ax.spines[['top', 'right']].set_visible(False)
ax.set_title(f"{int(keep.sum())} patients with at least {MIN_MAL} malignant cells "
             f"({counts.loc[keep, 'malignant'].sum():,} cells)", fontsize=10)
ax.legend(handles=[
    Patch(color=STATUS_COLORS['malignant'], label=f'malignant (kept, >= {MIN_MAL})'),
    Patch(color=EXCLUDED_COLOR, label=f'malignant (excluded, < {MIN_MAL})'),
    Patch(color=STATUS_COLORS['borderline'], label='borderline'),
    Patch(color=STATUS_COLORS['non_malignant'], label='non-malignant'),
    Patch(facecolor=STATUS_COLORS['not_tested'], hatch='////', edgecolor='white', label='not tested')],
    frameon=False, fontsize=8, loc='upper left', bbox_to_anchor=(1.0, 1.0))
fig.savefig(os.path.join(FIG_DIR, 'malignant_per_patient.png'))
plt.show()

## Downstream use of borderline and not_tested

03_2_subsetting keeps only malignant cells, so neither status enters the tumor object. not_tested means no evidence (B/plasma cells, uncovered cohorts, epithelium of gated-out cohorts) and must not be merged with non_malignant, which is a measurement. Borderline counts the cells where the two criteria disagree; a large borderline set would mean the subcluster cuts are badly placed.

Both still go through recelltypist_nonmalignant.py and get a normal-breast label, with the status kept next to it, in cell_annotation.csv.